# Exercise 7 — PYNQ-Z2 hardware tests

Run this notebook in Jupyter on the **PYNQ-Z2**, booted with the PYNQ image.
This notebook was not executed on the board during its preparation.

Copy both `notebooks/` (including `register_control.py`) and `overlay/`,
containing `registers.bit` and `registers.hwh` with matching base names.
Loading the overlay below programs the PL and replaces the previous overlay.
The assertions test the hardware directly; no software model substitutes for it.

Channel 1 (9 bits, output) carries `d[3:0]`, serial input (4), mode (5),
load_request (6), shift_request (7), and reset_request (8).
Channel 2 (4 bits, input) reads `q`.

Only one session may write to this GPIO. The adapter has no ACK:
a request must be sampled low before another rising transition.
The driver holds the data stable and performs eight AXI reads after each
complete write. These transactions allow the required PL cycles to elapse,
using standard uncached PYNQ MMIO and an active shared clock.
This mechanism is neither an acknowledgement nor a concurrent access protocol.


In [ ]:
from pathlib import Path
from pynq import Overlay
from pynq.lib import AxiGPIO
from register_control import RegisterControl, DATA_MASK, SERIAL, MODE, LOAD, SHIFT, RESET

BITFILE = Path('../overlay/registers.bit').resolve()
assert BITFILE.is_file(), f'Missing bitstream: {BITFILE}'
assert BITFILE.with_suffix('.hwh').is_file(), 'Matching HWH file is missing'
overlay = Overlay(str(BITFILE))
print('IP discovered in the HWH:', list(overlay.ip_dict))

# The actual name and parameters come from the HWH; no address is hardcoded.
def matches_contract(info):
    if 'axi_gpio' not in info.get('type', ''):
        return False
    parameters = info.get('parameters', {})
    return all(int(str(parameters.get(key, '-1')), 0) == expected
               for key, expected in [('C_IS_DUAL', 1), ('C_GPIO_WIDTH', 9),
                                     ('C_GPIO2_WIDTH', 4), ('C_ALL_OUTPUTS', 1),
                                     ('C_ALL_INPUTS_2', 1)])

candidates = [name for name, info in overlay.ip_dict.items() if matches_contract(info)]
assert len(candidates) == 1, f'Expected exactly one compatible GPIO: {candidates}. Check the HWH.'
gpio_name = candidates[0]
gpio = AxiGPIO(overlay.ip_dict[gpio_name])
control = RegisterControl(gpio)
print('Using GPIO:', gpio_name)
assert control.states() == (0, 0)


## Parallel loading and output selection
All 16 values must be stored correctly; changing the mode must not change either register's state.


In [ ]:
control.reset()
for value in range(16):
    control.load(value)
    assert control.states() == (value, 0), (value, control.states())
assert control.select(0) == 15
assert control.select(1) == 0
assert control.select(0) == 15
print('Loading all 16 values and output selection: OK')


## SIPO: shifting and bit insertion
The bit enters at `q(0)` and moves toward `q(3)`.


In [ ]:
control.reset()
control.load(10)
for bit, expected in [(1, 1), (0, 2), (0, 4), (0, 8), (0, 0)]:
    control.shift_bit(bit)
    assert control.states() == (10, expected)
for bit, expected in [(1, 1), (0, 2), (1, 5), (1, 11)]:
    control.shift_bit(bit)
    assert control.states() == (10, expected)
print('Shifting, shifting the bit out, and the 1011 sequence: OK')


## Held requests and value retention
A request held high produces a single operation, even across many transactions.
The data changes below occur after the operation has been executed.


In [ ]:
control.reset()
base = 10
control.write_command(base)
control.write_command(base | LOAD)
assert control.states() == (10, 0)
control.write_command(3 | LOAD)  # new data, request still held high.
for _ in range(16):
    assert control.states() == (10, 0)
control.write_command(3)  # rearm the requests.

base = 3 | SERIAL
control.write_command(base)
control.write_command(base | SHIFT)
for _ in range(16):
    assert control.states() == (10, 1)  # must not become 1111.
control.write_command(base)
for value in (0, 15, 6):
    control.write_command(value)  # requests low, changing data.
    assert control.states() == (10, 1)
print('Held requests and value retention: OK')


## Simultaneous commands, independent states, and reset
Reset clears both states and inhibits requests that are already high when reset is released.


In [ ]:
control.reset()
control.operate(data=5, serial=1, load=True, shift=True)
assert control.states() == (5, 1)
control.operate(data=12, serial=0, load=True, shift=True)
assert control.states() == (12, 2)
control.shift_bit(1)
assert control.states() == (12, 5)
control.load(9)
assert control.states() == (9, 5)

base = 15 | SERIAL | LOAD | SHIFT
control.write_command(base | RESET)
assert control.states() == (0, 0)
control.write_command(base)  # requests held high while reset is released.
for _ in range(8):
    assert control.states() == (0, 0)
control.write_command(0)  # rearming is required.
control.operate(data=7, serial=1, load=True, shift=True)
assert control.states() == (7, 1)
control.reset()
assert control.states() == (0, 0)
print('Simultaneous commands, independent states, reset, and recovery: OK')
print('All notebook hardware tests passed.')


Save the notebook with its outputs after running it on the board. The final
message is evidence of hardware validation only if it comes from that board execution.
This notebook does not measure setup/hold or recovery/removal timing: review the
post-route Vivado reports before programming the PL.
